# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


### Lectura de la imagen

Con la convención `(row, col)` empezando en `(0, 0)` arriba a la izquierda:

| Elemento | Celdas |
|---|---|
| `START` | `(0,0)` |
| Estanterías (paredes) | `(0,3)`, `(1,1)`, `(2,4)`, `(4,2)` |
| Piso resbaloso | `(1,2)`, `(2,1)`, `(3,3)` |
| Entrega `+10` (terminal) | `(0,5)` |
| Carga `+2` (terminal) | `(2,2)` |
| Peligro mortal `-10` (terminal) | `(3,5)` |
| Peligros `-3` (no terminales) | `(1,4)`, `(4,1)` |

Total: 30 celdas - 4 estanterías = **26 estados**.

In [1]:
import numpy as np

class WarehouseMDP:
    # Agregamos parámetros con valores por defecto para poder hacer
    # los experimentos de la Parte 5 sin reescribir la clase.
    def __init__(self, living_reward=-1.0, gamma=0.9, p_slip_intended=0.60):
        self.height = 5
        self.width = 6

        # Posición inicial del robot (esquina superior izquierda)
        self.start = (0, 0)

        # Estanterías: el robot no puede entrar a estas celdas
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}

        # Celdas amarillas de piso resbaloso
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        self.terminal_states = {
            (0, 5): +10.0,   # zona de entrega
            (2, 2): +2.0,    # estación de carga
            (3, 5): -10.0,   # peligro mortal
        }

        # Peligros NO terminales: penalizan, pero el robot sigue moviéndose
        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = living_reward
        self.gamma = gamma

        # Probabilidades (deseada, desviación izquierda, desviación derecha)
        self.p_normal = (0.90, 0.05, 0.05)
        p_dev = (1 - p_slip_intended) / 2   # lo que sobra se reparte en las dos desviaciones
        self.p_slippery = (p_slip_intended, p_dev, p_dev)

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        # Válido = dentro del grid y que no sea una estantería
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False
        return state not in self.walls

    def states(self):
        # Recorremos todas las celdas y nos quedamos con las válidas
        return [
            (row, col)
            for row in range(self.height)
            for col in range(self.width)
            if self.is_valid_state((row, col))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        # R(s): terminal -> su recompensa, peligro -> -3, otro -> costo por paso
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        # Un terminal es un estad final: se queda ahí con probabilidad 1
        if self.is_terminal(state):
            return [(state, 1.0)]

        # Las probabilidades dependen del piso del estado ACTUAL
        if state in self.slippery_states:
            p_int, p_left, p_right = self.p_slippery
        else:
            p_int, p_left, p_right = self.p_normal

        dr, dc = action
        # Girar la acción 90° a la izquierda y 90° a la derecha
        # (ej: si la acción es UP, izquierda = LEFT y derecha = RIGHT)
        left = (-dc, dr)
        right = (dc, -dr)

        outcomes = [(action, p_int), (left, p_left), (right, p_right)]

        transitions = []
        for move, prob in outcomes:
            next_state = (state[0] + move[0], state[1] + move[1])
            # Si se sale del grid o choca con una estantería, se queda quieto
            if not self.is_valid_state(next_state):
                next_state = state
            transitions.append((next_state, prob))

        return transitions


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    # sum_{s'} T(s,a,s') V(s')
    # Multiplicamos el valor de cada posible estado siguiente por su probabilidad
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    # Inicializamos V_0(s) = 0 en todos los estados
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        # Copia para actualizar todos los estados usando solo V_k
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                # En un terminal el valor es solo su recompensa
                V_new[state] = grid.get_reward(state)
            else:
                # max_a sum T(s,a,s') V_k(s')
                best_expected_value = max(
                    expected_next_value(grid, state, action, V)
                    for action in grid.actions
                )
                # Bellman: R(s) + gamma * mejor valor esperado
                V_new[state] = grid.get_reward(state) + grid.gamma * best_expected_value

            # Guardamos el cambio más grande de esta iteración
            biggest_change = max(biggest_change, abs(V_new[state] - V[state]))

        V = V_new

        # Criterio de parada: ningún estado cambió más que theta
        if biggest_change < threshold:
            break

    return V, iteration + 1


def extract_policy(grid, V):
    # pi*(s) = argmax_a sum T(s,a,s') V(s')
    policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue  # en los terminales no hay que decidir nada
        # max con key devuelve la ACCIÓN que da el mayor valor esperado
        policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )
    return policy


## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    # Calcula V^pi: igual que Value Iteration pero SIN max,
    # porque la acción ya la decide la política.
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                action = policy[state]   # acción que dice la política
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * expected_next_value(grid, state, action, V)
                )

            biggest_change = max(biggest_change, abs(V_new[state] - V[state]))

        V = V_new
        if biggest_change < threshold:
            break

    return V, iteration + 1


def policy_improvement(grid, V):
    # Para cada estado escogemos la acción greedy respecto a V^pi
    new_policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        new_policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )
    return new_policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. política inicial arbitraria: UP en todos los estados no terminales
    policy = {
        state: (-1, 0)
        for state in grid.states()
        if not grid.is_terminal(state)
    }
    history = []

    for iteration in range(max_iter):
        # 2. evaluación de la política actual
        V, eval_iterations = policy_evaluation(grid, policy, threshold)

        # 3. mejora de la política
        new_policy = policy_improvement(grid, V)

        # Contamos cuántos estados cambiaron de acción
        changed = sum(new_policy[s] != policy[s] for s in policy)
        history.append({
            "policy_iteration": iteration + 1,
            "evaluation_sweeps": eval_iterations,
            "changed_actions": changed,
        })

        policy = new_policy

        # 4. si nada cambió, la política es estable -> óptima
        if changed == 0:
            break

    # Valores finales de la política óptima
    V, _ = policy_evaluation(grid, policy, threshold)
    return policy, V, history


## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [{'policy_iteration': 1, 'evaluation_sweeps': 89, 'changed_actions': 16}, {'policy_iteration': 2, 'evaluation_sweeps': 77, 'changed_actions': 5}, {'policy_iteration': 3, 'evaluation_sweeps': 22, 'changed_actions': 1}, {'policy_iteration': 4, 'evaluation_sweeps': 22, 'changed_actions': 0}]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +

## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Respuestas

**1.** Desde `START (0,0)` la política va `→ → ↓ ↓`: pasa por `(0,1)`, `(0,2)`, `(1,2)` y termina en la **estación de carga +2** `(2,2)`. No intenta llegar a la entrega +10.

**2.** Porque cada paso cuesta `-1` y además se descuenta con $\gamma=0.9$. La carga está a 4 pasos, mientras que la entrega está a unos 7 pasos y el camino pasa cerca del peligro `-3` en `(1,4)`. Al sumar los costos de todos esos pasos y descontar el +10 (que llega muy tarde), el valor esperado de ir a la entrega termina siendo menor que el de quedarse con el +2 cercano. Una recompensa más pequeña pero segura y cercana puede valer más que una grande y lejana.

**3.** Comparamos la política con una versión del mundo sin piso resbaloso (ver celda de abajo). Las diferencias aparecen en:
- `(1,2)` (que es resbaloso): con piso normal el robot iría `→` hacia la entrega; como el piso resbala (solo 60 % de ir a donde quiere), prefiere `↓` para caer en la carga, que está justo al lado.
- `(3,1)`: sin piso resbaloso subiría `↑` por `(2,1)`; como `(2,1)` es resbaloso, prefiere ir `→` y entrar a la carga por debajo desde `(3,2)`.

**4.** El costo por paso `-1` castiga los caminos largos: obliga al robot a terminar rápido. Como es grande comparado con las recompensas, hace que el robot prefiera la meta más cercana (+2) en vez de la lejana (+10).

**5.** Porque la dinámica depende del tipo de piso del estado actual: en piso normal las probabilidades son 0.90/0.05/0.05 y en piso resbaloso son 0.60/0.20/0.20. Por eso `get_transition_probs` primero revisa si `state` está en `slippery_states` antes de armar la distribución.

In [7]:
# Comparación: mismo mundo pero sin piso resbaloso
grid_normal = WarehouseMDP()
grid_normal.slippery_states = set()   # quitamos todas las celdas resbalosas

V_normal, _ = value_iteration(grid_normal)
pi_normal = extract_policy(grid_normal, V_normal)

print("Estados donde cambia la acción por culpa del piso resbaloso:")
for s in pi_vi:
    if pi_vi[s] != pi_normal[s]:
        print(f"  {s}: con resbaloso {ARROWS[pi_vi[s]]}  |  sin resbaloso {ARROWS[pi_normal[s]]}")

Estados donde cambia la acción por culpa del piso resbaloso:
  (1, 2): con resbaloso ↓  |  sin resbaloso →
  (3, 1): con resbaloso →  |  sin resbaloso ↑


In [8]:
# Función auxiliar: sigue la acción de la política desde START
# (sin ruido) para ver a qué terminal intenta llegar el robot.
def camino_desde_start(grid, policy, max_pasos=30):
    s = grid.start
    camino = [s]
    while not grid.is_terminal(s) and len(camino) < max_pasos:
        a = policy[s]
        siguiente = (s[0] + a[0], s[1] + a[1])
        if not grid.is_valid_state(siguiente):
            siguiente = s
        s = siguiente
        camino.append(s)
    return camino


def experimento(titulo, **parametros):
    grid_exp = WarehouseMDP(**parametros)
    V_exp, n_exp = value_iteration(grid_exp)
    pi_exp = extract_policy(grid_exp, V_exp)
    print(f"=== {titulo} ===  ({n_exp} iteraciones)")
    print_policy(grid_exp, pi_exp)
    camino = camino_desde_start(grid_exp, pi_exp)
    print("Camino desde START:", camino)
    print("Terminal al que llega:", camino[-1], "-> recompensa", grid_exp.get_reward(camino[-1]))
    return grid_exp, V_exp, pi_exp

print("Política original:")
print("Camino desde START:", camino_desde_start(grid, pi_vi))

Política original:
Camino desde START: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)]


### Experimento A — Menos costo por paso

**Predicción:** si moverse cuesta casi nada (`-0.1`), ya no importa tanto que el camino sea largo, así que el robot debería ir por la entrega +10.

In [9]:
grid_A, V_A, pi_A = experimento("Experimento A: living_reward = -0.1", living_reward=-0.1)

=== Experimento A: living_reward = -0.1 ===  (26 iteraciones)
 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 
Camino desde START: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4), (1, 5), (0, 5)]
Terminal al que llega: (0, 5) -> recompensa 10.0


**Resultado:** se confirma la predicción. Desde `START` el robot va `→ → ↓ → → → ↑` hasta la entrega +10. Ahora en `(1,2)` elige `→` en lugar de `↓`, y los valores de casi todos los estados pasan a ser positivos.

### Experimento B — Piso muy resbaloso

**Predicción:** el piso resbaloso se vuelve más peligroso, así que el robot debería evitarlo aún más y seguir yendo a la carga. Los valores de las celdas resbalosas deberían bajar.

In [10]:
grid_B, V_B, pi_B = experimento("Experimento B: piso resbaloso 0.40 / 0.30 / 0.30", p_slip_intended=0.40)
print("\nValores:")
print_values(grid_B, V_B)

=== Experimento B: piso resbaloso 0.40 / 0.30 / 0.30 ===  (22 iteraciones)
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 
Camino desde START: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)]
Terminal al que llega: (2, 2) -> recompensa 2.0

Valores:
 -2.706 |  -1.809 |  -0.797 |   WALL   |  +7.607 | +10.000
 -2.652 |   WALL   |  +0.395 |  +2.104 |  +3.670 |  +7.607
 -1.744 |  -0.670 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.831 |  -0.774 |  +0.534 |  -1.137 |  -2.152 | -10.000
 -2.785 |  -3.929 |   WALL   |  -2.152 |  -2.974 |  -4.041


**Resultado:** la política queda **igual** a la original y el robot sigue yendo a la carga +2. Lo que cambia son los valores: bajan en las celdas resbalosas y en las que están cerca de ellas (por ejemplo `(1,2)` pasa de ≈ +0.56 a ≈ +0.39 y `(3,3)` de ≈ -0.78 a ≈ -1.14), porque llegar a donde se quiere es menos seguro.

### Experimento C — Más paciencia

**Predicción:** con $\gamma=0.99$ el futuro casi no se descuenta, así que el +10 lejano debería valer más y el robot podría preferir la entrega.

In [11]:
grid_C, V_C, pi_C = experimento("Experimento C: gamma = 0.99", gamma=0.99)

=== Experimento C: gamma = 0.99 ===  (24 iteraciones)
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 
Camino desde START: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4), (1, 5), (0, 5)]
Terminal al que llega: (0, 5) -> recompensa 10.0


**Resultado:** sí. Con $\gamma = 0.99$ la política desde `START` cambia y va hasta la **entrega +10**: en `(1,2)` ahora elige `→`. Al ser más "paciente", el robot valora más la recompensa grande aunque esté lejos.

### Bonus — ¿Desde qué `living_reward` cambia la decisión?

Probamos varios valores de `living_reward` y miramos a qué terminal llega el robot desde `START`.

In [12]:
for lr in np.arange(-1.0, -0.49, 0.05):
    g = WarehouseMDP(living_reward=lr)
    V_tmp, _ = value_iteration(g)
    pi_tmp = extract_policy(g, V_tmp)
    final = camino_desde_start(g, pi_tmp)[-1]
    destino = "entrega +10" if final == (0, 5) else "carga +2"
    print(f"living_reward = {lr:+.2f}  ->  {destino}")

# Afinamos entre -0.80 y -0.75
print()
for lr in np.arange(-0.80, -0.749, 0.005):
    g = WarehouseMDP(living_reward=lr)
    V_tmp, _ = value_iteration(g)
    pi_tmp = extract_policy(g, V_tmp)
    final = camino_desde_start(g, pi_tmp)[-1]
    destino = "entrega +10" if final == (0, 5) else "carga +2"
    print(f"living_reward = {lr:+.3f}  ->  {destino}")

living_reward = -1.00  ->  carga +2
living_reward = -0.95  ->  carga +2


living_reward = -0.90  ->  carga +2
living_reward = -0.85  ->  carga +2
living_reward = -0.80  ->  carga +2
living_reward = -0.75  ->  entrega +10
living_reward = -0.70  ->  entrega +10
living_reward = -0.65  ->  entrega +10
living_reward = -0.60  ->  entrega +10
living_reward = -0.55  ->  entrega +10
living_reward = -0.50  ->  entrega +10

living_reward = -0.800  ->  carga +2
living_reward = -0.795  ->  entrega +10
living_reward = -0.790  ->  entrega +10
living_reward = -0.785  ->  entrega +10
living_reward = -0.780  ->  entrega +10
living_reward = -0.775  ->  entrega +10
living_reward = -0.770  ->  entrega +10
living_reward = -0.765  ->  entrega +10
living_reward = -0.760  ->  entrega +10
living_reward = -0.755  ->  entrega +10
living_reward = -0.750  ->  entrega +10


**Conclusión del bonus:** el cambio ocurre aproximadamente en **`living_reward ≈ -0.80`**.

- Con `living_reward ≤ -0.80` el robot prefiere ir a la **carga +2**.
- Con `living_reward > -0.80` (por ejemplo `-0.795`) ya intenta llegar a la **entrega +10**.